# 🏥 Viettel Race 2026 — Qwen 2.5 7B Inference Notebook

**Mục tiêu:** Chạy inference NER y khoa tiếng Việt với Qwen 2.5 7B (LoRA Epoch 1) trên 100 file input BTC.

## ✅ Checklist trước khi chạy
1. Runtime: **GPU** → `Runtime → Change runtime type → T4 GPU`
2. 100 file `.txt` của BTC đã có trên Drive
3. Thư mục LoRA adapter đã có trên Drive

> ⚡ Ước tính: ~15–25 phút trên T4 | ~5–8 phút trên A100

## Bước 1: Kiểm tra GPU & Cài đặt thư viện

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cài Unsloth (optimized 4-bit inference) + dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q rapidfuzz peft accelerate bitsandbytes tqdm
print("✅ Cài đặt xong!")


## Bước 2: Mount Google Drive & Thiết lập paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ================================================================
# ⚙️  CẤU HÌNH — Chỉnh các đường dẫn này trước khi chạy!
# ================================================================

# Thư mục adapter trên Drive (chứa adapter_config.json)
ADAPTER_PATH = "/content/drive/MyDrive/qwen2.5-7b-lora-adapter/qwen2.5-7b-lora-adapter"

# Thư mục input BTC (100 file .txt)
INPUT_DIR = "/content/drive/MyDrive/ViettelRace/input_turn2_vong1/input"

# Thư mục chứa hybrid_linker.py + db/medical_codes.db
REPO_PATH = "/content/drive/MyDrive/ViettelRace/Viettel_Race_2026"

# Output (tự tạo)
OUTPUT_DIR = "/content/output_qwen"

# ================================================================

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

assert Path(ADAPTER_PATH).exists(), f"❌ Adapter không tồn tại: {ADAPTER_PATH}"
assert Path(INPUT_DIR).exists(),    f"❌ Input dir không tồn tại: {INPUT_DIR}"

input_files = sorted(Path(INPUT_DIR).glob("*.txt"))
print(f"✅ Adapter: {ADAPTER_PATH}")
print(f"✅ Input: {len(input_files)} files trong {INPUT_DIR}")
print(f"✅ Output: {OUTPUT_DIR}")


## Bước 3: Load Model + LoRA Adapter

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit"
MAX_SEQ_LEN = 4096

print("⏳ Loading base model (4-bit quantized)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # auto: bfloat16 (A100) hoặc float16 (T4)
    load_in_4bit=True,
)

print("⏳ Loading LoRA adapter...")
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

# Inference mode — tắt gradient, tối ưu tốc độ
FastLanguageModel.for_inference(model)

print("\n✅ Model sẵn sàng!")
print(f"   VRAM đã dùng: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


## Bước 4: Prompt Template & Output Parser

In [ ]:
import json, re

SYSTEM_PROMPT = """Bạn là chuyên gia phân tích văn bản y khoa tiếng Việt.
Nhiệm vụ: Trích xuất TẤT CẢ thực thể y khoa từ đoạn văn bản.

Loại thực thể:
- CHẨN_ĐOÁN: Tên bệnh, chẩn đoán (tăng huyết áp, đái tháo đường, viêm phổi...)
- THUỐC: Tên thuốc, hoạt chất (Aspirin, Metformin, Amlodipine 5mg...)
- TRIỆU_CHỨNG: Biểu hiện lâm sàng (sốt, ho, khó thở, đau ngực...)
- TÊN_XÉT_NGHIỆM: Tên xét nghiệm (X-quang, siêu âm, ECG, HbA1c...)
- KẾT_QUẢ_XÉT_NGHIỆM: Giá trị xét nghiệm (Glucose 12.5 mmol/L, Hb 9.2...)

Assertions (nếu có):
- isHistorical: bệnh/thuốc đã có trong quá khứ
- isNegated: phủ định (không có, loại trừ)
- isHypothetical: nghi ngờ, chưa chắc

Quy tắc:
1. position = [start, end] — vị trí ký tự CHÍNH XÁC trong văn bản gốc (end = start + len(text))
2. Chỉ trả về JSON array, không giải thích thêm

Format:
[
  {"text": "tên thực thể", "position": [start, end], "type": "LOẠI", "assertions": [], "candidates": []},
  ...
]
Nếu không có thực thể: []"""

def build_prompt(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Phân tích văn bản y khoa sau:\n\n{text}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_output(raw, source_text):
    """Parse JSON từ model output, auto-fix position nếu sai."""
    raw = raw.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw).strip()
    m = re.search(r'(\[.*\])', raw, re.DOTALL)
    if m: raw = m.group(1)

    try:
        entities = json.loads(raw)
    except:
        return []
    if not isinstance(entities, list):
        return []

    valid_types = {"CHẨN_ĐOÁN", "THUỐC", "TRIỆU_CHỨNG", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"}
    valid_assertions = {"isHistorical", "isNegated", "isHypothetical"}
    cleaned = []

    for ent in entities:
        if not isinstance(ent, dict): continue
        text_val = str(ent.get("text", "")).strip()
        etype = ent.get("type", "")
        if not text_val or etype not in valid_types: continue

        pos = ent.get("position", [])
        if isinstance(pos, list) and len(pos) == 2:
            s, e = pos
            if isinstance(s, int) and isinstance(e, int):
                # Validate position
                if not (0 <= s < e <= len(source_text)) or source_text[s:e].lower() != text_val.lower():
                    idx = source_text.lower().find(text_val.lower())
                    pos = [idx, idx + len(text_val)] if idx >= 0 else [s, e]
        else:
            idx = source_text.lower().find(text_val.lower())
            pos = [idx, idx + len(text_val)] if idx >= 0 else [0, len(text_val)]

        assertions = [a for a in ent.get("assertions", []) if a in valid_assertions]
        cleaned.append({
            "text": text_val,
            "position": pos,
            "type": etype,
            "assertions": assertions,
            "candidates": []
        })

    return cleaned

print("✅ Prompt template và parser sẵn sàng")


## Bước 5: Setup HybridLinker (ICD-10 / RxNorm)

In [ ]:
import sys
sys.path.insert(0, f"{REPO_PATH}/pipeline")

try:
    from hybrid_linker import HybridLinker
    DB_PATH = f"{REPO_PATH}/db/medical_codes.db"
    linker = HybridLinker(db_path=DB_PATH, use_semantic=False)
    print("✅ HybridLinker loaded!")
except Exception as e:
    print(f"⚠️ HybridLinker không load: {e}")
    print("   → Inference sẽ không có entity linking (candidates = [])")
    linker = None


## Bước 6: Chạy Inference 100 files

In [ ]:
import time
from tqdm.auto import tqdm

# --- Cấu hình generation ---
GENERATION_CONFIG = dict(
    max_new_tokens=2048,
    temperature=0.1,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
)
MAX_INPUT_CHARS = 8000  # Giảm xuống 4000 nếu bị OOM

def infer_one(text):
    if len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS]
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, **GENERATION_CONFIG)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    entities = parse_output(raw, text)
    if linker:
        for e in entities:
            e["candidates"] = linker.link_entity(e["text"], e["type"])
    return entities

# --- Main loop ---
out_dir = Path(OUTPUT_DIR)
files = sorted(Path(INPUT_DIR).glob("*.txt"),
               key=lambda p: int(p.stem) if p.stem.isdigit() else 0)

stats = {"ok": 0, "err": 0, "ents": 0}
t0 = time.time()

for fpath in tqdm(files, desc="Inference"):
    out_path = out_dir / f"{fpath.stem}.json"
    if out_path.exists():  # Resume-friendly
        stats["ok"] += 1
        continue
    try:
        text = fpath.read_text(encoding="utf-8")
        ents = infer_one(text)
        out_path.write_text(json.dumps(ents, ensure_ascii=False, indent=2), encoding="utf-8")
        stats["ok"] += 1
        stats["ents"] += len(ents)
    except Exception as ex:
        print(f"❌ {fpath.name}: {ex}")
        out_path.write_text("[]")
        stats["err"] += 1

    if (stats["ok"] + stats["err"]) % 10 == 0:
        elapsed = time.time() - t0
        done = stats["ok"] + stats["err"]
        avg = elapsed / max(done, 1)
        eta = avg * (len(files) - done)
        print(f"  [{done}/{len(files)}] {avg:.1f}s/file | ETA {eta/60:.1f}min | Entities: {stats['ents']}")

print(f"\n{'='*60}")
print(f"✅ XONG! OK={stats['ok']} | ERR={stats['err']} | Entities={stats['ents']}")
print(f"   Thời gian: {(time.time()-t0)/60:.1f} phút")


## Bước 7: Kiểm tra & Download kết quả

In [ ]:
from collections import Counter

out_files = sorted(out_dir.glob("*.json"))
print(f"Files output: {len(out_files)}/100")

type_cnt = Counter()
total_e, linked_e = 0, 0
for fp in out_files:
    for e in json.loads(fp.read_text(encoding="utf-8")):
        type_cnt[e.get("type","?")] += 1
        total_e += 1
        if e.get("candidates"): linked_e += 1

print(f"\n📊 Tổng: {total_e} entities (avg {total_e/max(len(out_files),1):.1f}/file)")
print(f"   Có ICD/RxNorm: {linked_e} ({100*linked_e/max(total_e,1):.1f}%)")
print("\n   Phân bố loại:")
for k, v in type_cnt.most_common():
    print(f"     {k:30s}: {v}")

print("\n--- Preview 1.json ---")
print(json.dumps(json.loads((out_dir/'1.json').read_text(encoding='utf-8'))[:3], ensure_ascii=False, indent=2))


In [ ]:
import shutil
from google.colab import files

# Nén
zip_path = "/content/submission_qwen.zip"
shutil.make_archive("/content/submission_qwen", "zip", str(out_dir))
print(f"✅ Đã tạo: {zip_path}")

# Download
files.download(zip_path)
print("⬇️ Đang download submission_qwen.zip ...")


In [ ]:
# Backup lên Drive (optional)
BACKUP = f"/content/drive/MyDrive/ViettelRace/submissions/qwen_submission"
shutil.copytree(str(out_dir), BACKUP, dirs_exist_ok=True)
print(f"✅ Backed up → {BACKUP}")


---
## 🔧 Troubleshooting

| Lỗi | Giải pháp |
|-----|-----------|
| `CUDA out of memory` | Giảm `MAX_INPUT_CHARS = 4000`, hoặc dùng A100 |
| `Adapter not found` | Kiểm tra `ADAPTER_PATH` — trỏ đúng folder chứa `adapter_config.json` |
| JSON parse error nhiều | Giảm `temperature = 0.05` |
| Inference quá chậm | Dùng Colab Pro A100 (~3x nhanh hơn T4) |
| HybridLinker lỗi | Kiểm tra `REPO_PATH` trỏ đúng thư mục repo |